# Stream Processing Pipeline - NYC Yellow Taxi

Esse notebook roda o pipeline inteiro de ponta a ponta: baixa os dados reais, simula a chegada em stream, valida, agrega e grava o resultado em Parquet. Da pra rodar tanto local (Jupyter) quanto no Databricks sem alterar nada, o notebook detecta sozinho onde esta rodando.

Se o csv ainda nao estiver em `data/raw/`, o notebook baixa uma amostra da NYC TLC automaticamente (via NYC Open Data, ver celula de download mais abaixo). Precisa de internet na primeira execucao. Se preferir baixar antes e separado, tem o script `scripts/baixar_dataset.py` que faz a mesma coisa.

**Basta rodar todas as celulas em ordem (Run All).** Dependencias: ver `requirements.txt` (`pip install -r requirements.txt`). Local, e importante usar Java 8, 11 ou 17 (Spark 3.5 nao roda em Java mais novo que isso).

## Parte 1: Ingestao, validacao e output

### Setup

Garante que a versao do pyspark instalada e compativel (3.5.x, que roda em Java 8/11/17). Se a maquina tiver o pyspark 4.x instalado, ele exige Java 17+, entao aqui a gente forca a versao certa. No Databricks isso nao roda, o cluster ja vem com o Spark configurado.

In [0]:
import os
import sys
import subprocess

IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if not IS_DATABRICKS:
    precisa_instalar = False
    try:
        import pyspark
        if int(pyspark.__version__.split(".")[0]) >= 4:
            precisa_instalar = True  # pyspark 4.x exige o Java 17+, preferimos 3.5.x
    except ImportError:
        precisa_instalar = True

    if precisa_instalar:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pyspark==3.5.1", "pandas"])

### Sessao Spark

No Databricks a `spark` ja vem criada, `getOrCreate()` so reaproveita ela. Se der erro tipo `UnsupportedClassVersionError` rodando local, e porque a versao do Java instalada nao bate com a do pyspark (Spark 3.5.x precisa de Java 8, 11 ou 17). Se der erro mencionando `winutils`/`HADOOP_HOME`/`NativeIO`, e o problema conhecido do Spark local no Windows precisar do `winutils.exe`; costuma ser so aviso (WARN) e nao impede o pipeline de rodar, mas se travar mesmo, mais facil rodar em WSL, Linux/Mac ou direto no Databricks (foi assim que testamos).

In [0]:
from pyspark.sql import SparkSession

try:
    spark = SparkSession.builder.appName("nyc-taxi-streaming").getOrCreate()
except Exception as erro:
    print("Erro ao criar a sessao Spark.")
    print("UnsupportedClassVersionError => versao de Java incompativel (usar Java 8, 11 ou 17 pro Spark 3.5.x).")
    print("Mencao a winutils/HADOOP_HOME/NativeIO => problema conhecido do Spark local no Windows.")
    raise erro

### Ambiente: local x Databricks

Detecta automaticamente se esta rodando no Databricks (variavel de ambiente `DATABRICKS_RUNTIME_VERSION`) e escolhe os caminhos certos.

Historico do que ja tentamos aqui (documentando pra nao repetir erro):
- `dbfs:/FileStore/...`: falhou com `DBFS_DISABLED` (workspace com DBFS root publico desabilitado).
- `/tmp/...` (disco local generico): falhou com `LocalFilesystemAccessDeniedException` (esse workspace roda em modo Unity Catalog "Shared", que bloqueia acesso a disco local fora de `/Workspace`).
- `/Workspace/...` (pasta do proprio notebook): a leitura/escrita simples funcionou, mas o `writeStream` falhou no `_spark_metadata` com `CANNOT_LOAD_CHECKPOINT_FILE_MANAGER` -- o gerenciador de checkpoint do Databricks nao da suporte a `/Workspace` pra isso.
- Solucao atual: **Unity Catalog Volume**, que e o armazenamento pensado pra esse tipo de carga (leitura/escrita de arquivo, streaming, checkpoint) em clusters serverless/Unity Catalog. Criamos um Volume automaticamente (`CREATE VOLUME IF NOT EXISTS`) tentando os catalogos mais comuns em workspaces novos (`workspace`, `main`). Se nenhum dos dois existir ou voce nao tiver permissao neles, o notebook avisa e explica como achar um catalogo/schema que funcione no seu caso.

In [0]:
if IS_DATABRICKS:
    candidatos_catalogo = [("workspace", "default"), ("main", "default")]
    volume_ok = None
    erros = []
    for catalogo, schema in candidatos_catalogo:
        try:
            spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalogo}`.`{schema}`.`nyc_taxi_streaming`")
            volume_ok = (catalogo, schema)
            break
        except Exception as erro:
            erros.append(f"{catalogo}.{schema}: {erro}")

    if volume_ok is None:
        print("Nao consegui criar o Volume em nenhum catalogo padrao. Erros:")
        for e in erros:
            print(" -", e)
        print("Rode 'SHOW CATALOGS' e 'SHOW SCHEMAS IN <catalogo>' numa celula pra achar um catalogo/schema")
        print("onde voce tenha permissao de CREATE VOLUME, e ajuste CATALOG/SCHEMA abaixo manualmente.")
        CATALOG, SCHEMA = "workspace", "default"
    else:
        CATALOG, SCHEMA = volume_ok
        print(f"usando o volume {CATALOG}.{SCHEMA}.nyc_taxi_streaming")

    PYTHON_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/nyc_taxi_streaming"
    SPARK_BASE = PYTHON_BASE  # Volume usa o mesmo caminho pro Spark e pro Python, sem prefixo especial
else:
    PYTHON_BASE = "."
    SPARK_BASE = "."

RAW_PATH = f"{PYTHON_BASE}/data/raw/yellow_tripdata.csv"
STREAM_SOURCE_PATH_PY = f"{PYTHON_BASE}/data/stream_source"
STREAM_SOURCE_PATH_SPARK = f"{SPARK_BASE}/data/stream_source"
OUTPUT_PATH = f"{SPARK_BASE}/output/parquet"
CHECKPOINT_PATH = f"{SPARK_BASE}/output/checkpoints"

print("rodando no Databricks:" if IS_DATABRICKS else "rodando local:", IS_DATABRICKS)
print("PYTHON_BASE:", PYTHON_BASE)
print("RAW_PATH:", RAW_PATH)
print("STREAM_SOURCE_PATH_SPARK:", STREAM_SOURCE_PATH_SPARK)
print("OUTPUT_PATH:", OUTPUT_PATH)

usando o volume workspace.default.nyc_taxi_streaming
rodando no Databricks: True
PYTHON_BASE: /Volumes/workspace/default/nyc_taxi_streaming
RAW_PATH: /Volumes/workspace/default/nyc_taxi_streaming/data/raw/yellow_tripdata.csv
STREAM_SOURCE_PATH_SPARK: /Volumes/workspace/default/nyc_taxi_streaming/data/stream_source
OUTPUT_PATH: /Volumes/workspace/default/nyc_taxi_streaming/output/parquet


### Dados de entrada: download

A TLC hoje disponibiliza os arquivos oficiais so em Parquet, entao usamos a mesma base exportada em CSV pelo NYC Open Data (https://data.cityofnewyork.us), que e o proprio portal de dados da prefeitura de Nova York alimentado pela TLC. Baixamos uma amostra (primeira semana de janeiro/2023) direto pra `RAW_PATH`, se ainda nao existir.

In [0]:
import urllib.parse
import urllib.request

DATASET_BASE_URL = "https://data.cityofnewyork.us/resource/4b4i-vvec.csv"  # NYC Open Data - Yellow Taxi 2023
DATASET_PARAMS = {
    "$limit": "5000",
    "$where": "tpep_pickup_datetime between '2023-01-01T00:00:00' and '2023-01-07T23:59:59'",
    "$order": "tpep_pickup_datetime",
}

def baixar_dataset(destino):
    url = DATASET_BASE_URL + "?" + urllib.parse.urlencode(DATASET_PARAMS)
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    print(f"baixando dataset de {url}")
    urllib.request.urlretrieve(url, destino)
    print(f"salvo em {destino}")

if not os.path.exists(RAW_PATH):
    try:
        baixar_dataset(RAW_PATH)
    except Exception as erro:
        print("Nao foi possivel baixar o dataset.")
        print("Rode 'python scripts/baixar_dataset.py' antes, ou baixe manualmente:")
        print(DATASET_BASE_URL + "?" + urllib.parse.urlencode(DATASET_PARAMS))
        print(f"e salve em {RAW_PATH}")
        raise erro
else:
    print(f"usando o csv ja baixado em {RAW_PATH}")

usando o csv ja baixado em /Volumes/workspace/default/nyc_taxi_streaming/data/raw/yellow_tripdata.csv


### Schema dos dados

Definido na mao porque em streaming o Spark exige isso. Segue as colunas do csv atual do NYC Open Data/TLC (esquema pos-2016, com zonas `pulocationid`/`dolocationid` em vez de latitude/longitude).

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

schema = StructType([
    StructField("vendorid", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("ratecodeid", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("pulocationid", IntegerType(), True),
    StructField("dolocationid", IntegerType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True),
])

### Simulando a chegada dos dados

A TLC libera os dados em lote (arquivo fechado), entao pra simular um stream a gente parte o csv em pedacos menores e vai escrevendo eles aos poucos na pasta de origem. O Spark fica de olho nessa pasta com o `readStream`. Os valores padrao aqui (`intervalo_segundos=2`, `limite_arquivos=10`) sao pensados pra demo rapida, ajustar se quiser um teste maior.

In [0]:
import time
import pandas as pd

def simular_stream(raw_path, destino, linhas_por_arquivo=200, intervalo_segundos=2, limite_arquivos=10):
    os.makedirs(destino, exist_ok=True)
    leitor = pd.read_csv(raw_path, chunksize=linhas_por_arquivo)
    for i, chunk in enumerate(leitor):
        if limite_arquivos and i >= limite_arquivos:
            break
        arquivo = os.path.join(destino, f"corridas_{i:05d}.csv")
        chunk.to_csv(arquivo, index=False)
        print(f"gerado {arquivo}")
        time.sleep(intervalo_segundos)

### Ingestao (readStream) e validacao (bonus 1)

Descarta corrida sem passageiro, sem distancia, com tarifa negativa/zerada, e linha com algum campo fundamental nulo (o dataset tem alguns desses casos, e um bom teste real do filtro). A pasta de origem precisa existir antes do `readStream.load()`, por isso o `makedirs` logo no comeco.

In [0]:
from pyspark.sql import functions as F

os.makedirs(STREAM_SOURCE_PATH_PY, exist_ok=True)  # o readStream exige que a pasta ja exista

df_bruto = (
    spark.readStream
    .format("csv")
    .option("header", "true")
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss.SSS")
    .schema(schema)
    .option("maxFilesPerTrigger", 1)
    .load(STREAM_SOURCE_PATH_SPARK)
)

df_validado = (
    df_bruto
    .filter(F.col("passenger_count") > 0)
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") > 0)
    .dropna(subset=["vendorid", "tpep_pickup_datetime", "tpep_dropoff_datetime"])
)

### Output em Parquet

Aqui o `df_validado` e o ponto de entrada pra Parte 2 (agregacao com window). Por enquanto grava direto o resultado validado.

O trigger padrao do Structured Streaming fica rodando continuamente, verificando a pasta de tempos em tempos, mas isso deu erro `INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED` no nosso Databricks (workspace serverless, sem suporte a query continua). Por isso usamos `trigger(availableNow=True)`: ele processa tudo que estiver disponivel na pasta no momento do `.start()` e termina sozinho, sem precisar rodar em background nem dar `.stop()` manual. Por isso essa celula so monta a query (nao chama `.start()` ainda) -- disparamos ela na proxima celula, depois de gerar os arquivos.

In [0]:
query_writer = (
    df_validado.writeStream
    .format("parquet")
    .option("path", OUTPUT_PATH)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
)

### Rodando a simulacao

Primeiro geramos todos os arquivos simulando a chegada das corridas, depois disparamos a query (`query_writer.start()`). Com `availableNow=True` ela processa tudo que tiver na pasta nesse momento (um arquivo por vez, por causa do `maxFilesPerTrigger=1`) e termina sozinha.

In [0]:
simular_stream(RAW_PATH, STREAM_SOURCE_PATH_PY, linhas_por_arquivo=200, intervalo_segundos=2, limite_arquivos=10)

query = query_writer.start()
query.awaitTermination()  # com availableNow=True, termina sozinha quando acabar de processar tudo
print("query finalizada")

gerado /Volumes/workspace/default/nyc_taxi_streaming/data/stream_source/corridas_00000.csv
gerado /Volumes/workspace/default/nyc_taxi_streaming/data/stream_source/corridas_00001.csv
gerado /Volumes/workspace/default/nyc_taxi_streaming/data/stream_source/corridas_00002.csv
gerado /Volumes/workspace/default/nyc_taxi_streaming/data/stream_source/corridas_00003.csv
gerado /Volumes/workspace/default/nyc_taxi_streaming/data/stream_source/corridas_00004.csv
gerado /Volumes/workspace/default/nyc_taxi_streaming/data/stream_source/corridas_00005.csv
gerado /Volumes/workspace/default/nyc_taxi_streaming/data/stream_source/corridas_00006.csv
gerado /Volumes/workspace/default/nyc_taxi_streaming/data/stream_source/corridas_00007.csv
gerado /Volumes/workspace/default/nyc_taxi_streaming/data/stream_source/corridas_00008.csv
gerado /Volumes/workspace/default/nyc_taxi_streaming/data/stream_source/corridas_00009.csv
query finalizada


### Resultado esperado

Lendo de volta o parquet gravado, pra confirmar que o pipeline processou os dados.

In [0]:
resultado = spark.read.parquet(OUTPUT_PATH)
print("total de linhas gravadas em parquet:", resultado.count())
resultado.show(10)

total de linhas gravadas em parquet: 1868
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|vendorid|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|ratecodeid|store_and_fwd_flag|pulocationid|dolocationid|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2023-01-01 00:06:08|  2023-01-01 00:28:01|            1.0|         4.37|       1.0|                 N|          79|          33|      

## Parte 2: Agregacao, Window e Deploy

### Agregacao por janela de tempo (Window)

Com os dados validados, agrupamos as corridas em **janelas de 1 hora** pelo horario de embarque (`tpep_pickup_datetime`), separadas por tipo de pagamento. Para cada janela calculamos: total de corridas, media de tarifa, media de distancia e receita total.

O `withWatermark` de 10 minutos diz ao Spark por quanto tempo uma janela ainda pode receber dados atrasados. Depois desse prazo, a janela e fechada e o resultado e gravado. Sem o watermark, o `outputMode("append")` nao funciona com agregacoes em streaming.

In [0]:
AGGS_OUTPUT_PATH = f"{SPARK_BASE}/output/parquet_aggs"
AGGS_CHECKPOINT_PATH = f"{SPARK_BASE}/output/checkpoints_aggs"

# limpa tentativa anterior: checkpoint vazio impede reprocessamento dos arquivos
try:
    dbutils.fs.rm(AGGS_CHECKPOINT_PATH, True)
    dbutils.fs.rm(AGGS_OUTPUT_PATH, True)
except:
    import shutil, os as _os
    for _p in [AGGS_CHECKPOINT_PATH, AGGS_OUTPUT_PATH]:
        if _os.path.exists(_p):
            shutil.rmtree(_p)

df_por_janela = (
    df_validado
    .withWatermark("tpep_pickup_datetime", "10 minutes")
    .groupBy(
        F.window("tpep_pickup_datetime", "1 hour"),
        "payment_type"
    )
    .agg(
        F.count("*").alias("total_corridas"),
        F.round(F.avg("fare_amount"), 2).alias("media_tarifa"),
        F.round(F.avg("trip_distance"), 2).alias("media_distancia"),
        F.round(F.sum("total_amount"), 2).alias("receita_total")
    )
)

# foreachBatch + update: emite janelas atualizadas a cada micro-batch,
# sem esperar o watermark fechar a janela (evita resultado vazio com availableNow)
def gravar_batch(batch_df, batch_id):
    batch_df.write.mode("append").parquet(AGGS_OUTPUT_PATH)

query_aggs = (
    df_por_janela.writeStream
    .foreachBatch(gravar_batch)
    .option("checkpointLocation", AGGS_CHECKPOINT_PATH)
    .outputMode("update")
    .trigger(availableNow=True)
    .start()
)
query_aggs.awaitTermination()
print("query de agregacao finalizada")

query de agregacao finalizada


### Resultado das agregacoes

Cada linha aqui representa uma janela de 1 hora + tipo de pagamento, nao uma corrida individual. Espera-se um numero bem menor de linhas do que a saida da Parte 1.

In [0]:
from pyspark.sql.window import Window as WSpec

resultado_aggs = spark.read.parquet(AGGS_OUTPUT_PATH)

# com update mode a mesma janela pode aparecer em varios batches; pega o estado final (maior contagem)
w = WSpec.partitionBy("window", "payment_type").orderBy(F.desc("total_corridas"))
resultado_final = (
    resultado_aggs
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print("total de janelas de tempo gravadas:", resultado_final.count())
(
    resultado_final
    .select(
        F.col("window.start").alias("janela_inicio"),
        F.col("window.end").alias("janela_fim"),
        "payment_type",
        "total_corridas",
        "media_tarifa",
        "media_distancia",
        "receita_total"
    )
    .orderBy("janela_inicio", "payment_type")
    .show(30, truncate=False)
)

total de janelas de tempo gravadas: 4
+-------------------+-------------------+------------+--------------+------------+---------------+-------------+
|janela_inicio      |janela_fim         |payment_type|total_corridas|media_tarifa|media_distancia|receita_total|
+-------------------+-------------------+------------+--------------+------------+---------------+-------------+
|2023-01-01 00:00:00|2023-01-01 01:00:00|1           |1468          |19.15       |3.44           |42256.25     |
|2023-01-01 00:00:00|2023-01-01 01:00:00|2           |373           |20.21       |3.84           |9459.3       |
|2023-01-01 00:00:00|2023-01-01 01:00:00|3           |8             |20.68       |4.72           |201.65       |
|2023-01-01 00:00:00|2023-01-01 01:00:00|4           |19            |14.68       |2.41           |368.9        |
+-------------------+-------------------+------------+--------------+------------+---------------+-------------+

